# 08 空間流病 — 參考解答

松柏護理之家退伍軍人症群聚事件空間分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## 題目 1：致死率空間分布

In [ ]:
# 計算 floor × wing 致死率
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== 翼區侵襲率 & 致死率 ===")
print(spatial[["floor", "wing", "total", "infected", "died", "attack_rate", "cfr"]].to_string(index=False))

# 致死率熱力圖
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=ax)
ax.set_title("致死率 (%) by Floor \u00d7 Wing")
ax.set_ylabel("Floor")
plt.tight_layout()
plt.show()

# 解讀
highest_cfr = spatial.loc[spatial["cfr"].idxmax()]
highest_ar = spatial.loc[spatial["attack_rate"].idxmax()]
print(f"\n致死率最高：{highest_cfr['floor']}F-{highest_cfr['wing']}（{highest_cfr['cfr']}%）")
print(f"侵襲率最高：{highest_ar['floor']}F-{highest_ar['wing']}（{highest_ar['attack_rate']}%）")
print("\n\u2192 致死率最高的翼區不一定是侵襲率最高的翼區")
print("\u2192 致死率還受住民特性（年齡、共病）影響，不完全取決於暴露強度")

## 題目 2：淋浴使用的空間分布

In [ ]:
# 淋浴使用比例
shower = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    shower_users=("shower_use", "sum"),
    infected=("infected", "sum"),
).reset_index()
shower["shower_pct"] = (shower["shower_users"] / shower["total"] * 100).round(1)
shower["attack_rate"] = (shower["infected"] / shower["total"] * 100).round(1)

print("=== 淋浴比例 vs 侵襲率 ===")
print(shower[["floor", "wing", "shower_pct", "attack_rate"]].to_string(index=False))

# 並排熱力圖
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

hm_shower = shower.pivot(index="floor", columns="wing", values="shower_pct")
sns.heatmap(hm_shower, annot=True, fmt=".1f", cmap="Blues",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("淋浴使用比例 (%)")
axes[0].set_ylabel("Floor")

hm_ar = shower.pivot(index="floor", columns="wing", values="attack_rate")
sns.heatmap(hm_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("侵襲率 (%)")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

# 相關性
corr = shower[["shower_pct", "attack_rate"]].corr().iloc[0, 1]
print(f"\n淋浴比例 vs 侵襲率相關係數：r = {corr:.3f}")
print("\n\u2192 如果兩張熱力圖的高低分布類似，支持水源傳播假說")
print("\u2192 但也要考慮干擾因子（如 functional_status 影響淋浴能力，Ch05 已分析）")

## 題目 3（挑戰題）：高風險房間清單

In [ ]:
# 每間房侵襲率
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# 解析 floor 和 wing
room_stats["floor"] = room_stats["room"].str[0].astype(int)
room_stats["wing"] = room_stats["room"].str[1]

# 篩選 >= 75%
high_risk = (
    room_stats[room_stats["attack_rate"] >= 75]
    .sort_values("attack_rate", ascending=False)
    [["room", "total", "infected", "attack_rate", "floor", "wing"]]
)

print(f"=== 高風險房間清單（侵襲率 \u2265 75%）===")
print(f"共 {len(high_risk)} 間\n")
print(high_risk.to_string(index=False))

# 按翼區統計
print("\n=== 高風險房間的翼區分布 ===")
wing_counts = high_risk.groupby(["floor", "wing"]).size().reset_index(name="high_risk_rooms")
print(wing_counts.to_string(index=False))

print("\n\u2192 提交此清單給感控團隊，優先對這些房間進行環境採檢")
print("\u2192 特別關注高風險房間集中的翼區，檢查蓮蓬頭和熱水管線")

### 解讀

- **致死率 vs 侵襲率**：兩者不一定正相關。侵襲率反映暴露風險，致死率反映宿主脆弱度
- **淋浴 × 空間**：如果淋浴比例高的翼區也是侵襲率高的翼區，空間分析強化了水源傳播假說
- **高風險房間**：集中在特定翼區的高風險房間，提示該翼區的供水系統可能是傳播途徑
- **行動建議**：對高風險翼區的蓮蓬頭、熱水管線進行退伍軍人菌培養與環境採檢

## 題目 4 解答

In [ ]:
# 登革熱：各行政區病例（積水多的安南區風險偏高）
rng = np.random.default_rng(841)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
rate_per_100k = {"安南區": 22, "三民區": 8, "北屯區": 5, "板橋區": 4, "中西區": 9}
_recs = []
for d in districts:
    n = rng.poisson(rate_per_100k[d] * pop[d] / 100000)
    for _ in range(n):
        _recs.append({"district": d, "age": int(rng.integers(5, 85)),
                      "serotype": rng.choice(["DENV-1", "DENV-2", "DENV-3"])})
dengue = pd.DataFrame(_recs)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"登革熱通報 {len(dengue)} 例，橫跨 {dengue['district'].nunique()} 個行政區")

by_dist = dengue.groupby("district").size().reset_index(name="cases")
by_dist = by_dist.merge(region_pop, on="district")
by_dist["rate_per_100k"] = (by_dist["cases"] / by_dist["population"] * 100000).round(1)
by_dist = by_dist.sort_values("rate_per_100k", ascending=False)
print(by_dist.to_string(index=False))

top_rate = by_dist.iloc[0]
top_cases = by_dist.sort_values("cases", ascending=False).iloc[0]

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=by_dist, x="district", y="rate_per_100k", color="#D97757", ax=ax)
ax.set_title("各行政區登革熱發生率（每十萬人）")
ax.set_ylabel("發生率 / 10萬")
plt.tight_layout(); plt.show()

print(f"\n發生率最高：{top_rate['district']}（{top_rate['rate_per_100k']}/10萬，{top_rate['cases']} 例）")
print(f"病例數最多：{top_cases['district']}（{top_cases['cases']} 例，{top_cases['rate_per_100k']}/10萬）")
print("解讀：人口多的區病例多不代表風險高；用發生率（人口標準化）才能公平比較各區。")

## 題目 5 解答

In [ ]:
# COVID-19：4x5 網格區域的人口與病例（北部 A/B 列風險較高）
rng = np.random.default_rng(852)
_recs = []
for r in list("ABCD"):
    for c in range(1, 6):
        popn = int(rng.integers(2000, 6000))
        base = 0.03 + (0.05 if r in ("A", "B") else 0.0) + rng.normal(0, 0.008)
        cases = rng.binomial(popn, max(0.005, base))
        _recs.append({"region": f"{r}{c}", "row": r, "col": c, "population": popn, "cases": cases})
covid = pd.DataFrame(_recs)
print(f"COVID-19：{len(covid)} 個區域，總人口 {covid['population'].sum():,}，總病例 {covid['cases'].sum()}")

covid["attack_rate_pct"] = (covid["cases"] / covid["population"] * 100).round(2)
grid = covid.pivot(index="row", columns="col", values="attack_rate_pct")

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(grid, annot=True, fmt=".2f", cmap="Reds", ax=ax,
            cbar_kws={"label": "侵襲率 (%)"})
ax.set_title("各區域 COVID-19 侵襲率熱區圖")
plt.tight_layout(); plt.show()

hottest = covid.sort_values("attack_rate_pct", ascending=False).iloc[0]
north = covid[covid["row"].isin(["A", "B"])]["attack_rate_pct"].mean()
south = covid[covid["row"].isin(["C", "D"])]["attack_rate_pct"].mean()
print(f"侵襲率最高區域：{hottest['region']}（{hottest['attack_rate_pct']}%）")
print(f"北側(A/B)平均 {north:.2f}% vs 南側(C/D)平均 {south:.2f}% → 熱區集中在北側")

## 題目 6 解答

In [ ]:
# 腸病毒：國小各年級各班的學生數與病例（低年級風險較高）
rng = np.random.default_rng(863)
_recs = []
for g in range(1, 7):
    for cl in range(1, 6):
        students = int(rng.integers(25, 35))
        risk = max(0.02, 0.28 - g * 0.03)
        cases = rng.binomial(students, risk)
        _recs.append({"grade": g, "classroom": cl, "students": students, "cases": cases})
ev = pd.DataFrame(_recs)
print(f"腸病毒：{ev['grade'].nunique()} 個年級 × {ev['classroom'].nunique()} 班，共 {ev['cases'].sum()} 例")

ev["attack_rate_pct"] = (ev["cases"] / ev["students"] * 100).round(1)
mat = ev.pivot(index="grade", columns="classroom", values="attack_rate_pct")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(mat, annot=True, fmt=".1f", cmap="Reds", ax=ax,
            cbar_kws={"label": "侵襲率 (%)"})
ax.set_title("腸病毒 年級 × 班級 侵襲率熱區圖")
ax.set_xlabel("班級"); ax.set_ylabel("年級")
plt.tight_layout(); plt.show()

by_grade = ev.groupby("grade").apply(
    lambda g: 100 * g["cases"].sum() / g["students"].sum(), include_groups=False).round(1)
print("各年級整體侵襲率(%):")
print(by_grade.to_string())
print(f"\n最高：{by_grade.idxmax()} 年級（{by_grade.max()}%）；最低：{by_grade.idxmin()} 年級（{by_grade.min()}%）")
print("解讀：低年級侵襲率較高，符合腸病毒好發於幼童、且低年級衛生習慣與接觸型態的特性。")

## 題目 7 解答

In [ ]:
# 諾羅病毒：宴會 25 桌的座位平面圖（靠近海鮮區的桌次侵襲率高）
rng = np.random.default_rng(874)
_recs = []
for t in range(1, 26):
    x, y = (t - 1) % 5, (t - 1) // 5
    attendees = int(rng.integers(8, 12))
    near_seafood = (x <= 1 and y <= 1)   # 左下角靠海鮮區
    ar = 0.6 if near_seafood else 0.1
    cases = rng.binomial(attendees, ar)
    _recs.append({"table": t, "x": x, "y": y, "attendees": attendees, "cases": cases})
noro = pd.DataFrame(_recs)
print(f"諾羅病毒宴會：{len(noro)} 桌，{noro['attendees'].sum()} 人出席，{noro['cases'].sum()} 人發病")

noro["attack_rate"] = (noro["cases"] / noro["attendees"]).round(2)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(noro["x"], noro["y"], s=noro["attendees"] * 25,
                c=noro["attack_rate"], cmap="Reds", edgecolor="#555", linewidth=0.6)
for _, r in noro.iterrows():
    ax.annotate(str(r["table"]), (r["x"], r["y"]), ha="center", va="center", fontsize=7)
ax.set_title("宴會桌次諾羅病毒 spot map（點大小=人數、顏色=侵襲率）")
ax.set_xlabel("桌次 X"); ax.set_ylabel("桌次 Y"); ax.invert_yaxis()
plt.colorbar(sc, ax=ax, label="侵襲率"); plt.tight_layout(); plt.show()

cluster = noro[noro["attack_rate"] >= 0.4].sort_values("attack_rate", ascending=False)
print("高侵襲率群聚桌次：")
print(cluster[["table", "x", "y", "attendees", "cases", "attack_rate"]].to_string(index=False))
print("解讀：高侵襲率桌次集中在平面圖左下角，指向鄰近的海鮮／冷盤供應區為可能汙染源。")

## 題目 8 解答

In [ ]:
# 結核病：12 個鄉鎮的人口、擁擠指數與病例（人口差異大 → 小分母率不穩）
rng = np.random.default_rng(885)
_recs = []
for i in range(1, 13):
    popn = int(rng.integers(3000, 120000))
    crowding = round(float(rng.uniform(0.5, 2.0)), 2)
    cases = rng.poisson(15 * crowding * popn / 100000)
    _recs.append({"township": f"T{i:02d}", "population": popn,
                  "crowding_index": crowding, "cases": cases})
tb = pd.DataFrame(_recs)
print(f"結核病：{len(tb)} 個鄉鎮，人口 {tb['population'].min():,}–{tb['population'].max():,}")

tb["rate_per_100k"] = (tb["cases"] / tb["population"] * 100000).round(1)
by_cases = tb.sort_values("cases", ascending=False)["township"].head(3).tolist()
by_rate = tb.sort_values("rate_per_100k", ascending=False)["township"].head(3).tolist()
print(tb.sort_values("rate_per_100k", ascending=False).to_string(index=False))

small = tb.sort_values("population").head(3)
corr = tb["crowding_index"].corr(tb["rate_per_100k"])
print(f"\n病例數前三：{by_cases}")
print(f"發生率前三：{by_rate}  → 兩份名單{'相同' if by_cases == by_rate else '不同'}")
print(f"人口最小的鄉鎮：{small['township'].tolist()}（分母小，1–2 例即可使發生率大幅跳動 → 率不穩定）")
print(f"擁擠指數 vs 發生率相關 r = {corr:.2f} → 擁擠程度與結核發生率{'正相關' if corr > 0.3 else '關聯不明顯'}")
print("解讀：鄉鎮地圖建議同時呈現發生率（人口標準化）與病例數，並對小人口區的率加註信賴區間或合併，避免被極端率誤導。")